In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

path = "../data/raw/"

customers   = pd.read_csv(path + "olist_customers_dataset.csv")
geo         = pd.read_csv(path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments    = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(path + "olist_order_reviews_dataset.csv")
orders      = pd.read_csv(path + "olist_orders_dataset.csv")
products    = pd.read_csv(path + "olist_products_dataset.csv")
sellers     = pd.read_csv(path + "olist_sellers_dataset.csv")
categories  = pd.read_csv(path + "product_category_name_translation.csv")

print("All 9 tables loaded ✓")

All 9 tables loaded ✓


In [2]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns converted ✓")
print()
print(orders[date_columns].dtypes)

Date columns converted ✓

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [3]:
# Keep only delivered orders
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

# Confirm the filter worked
print(f"Original orders:     {len(orders):,}")
print(f"Delivered orders:    {len(orders_delivered):,}")
print(f"Removed:             {len(orders) - len(orders_delivered):,}")
print(f"Fulfilment rate:     {len(orders_delivered)/len(orders)*100:.1f}%")
print()
print("Order status of removed rows:")
print(orders[orders['order_status'] != 'delivered']['order_status'].value_counts())

Original orders:     99,441
Delivered orders:    96,478
Removed:             2,963
Fulfilment rate:     97.0%

Order status of removed rows:
order_status
shipped        1107
canceled        625
unavailable     609
invoiced        314
processing      301
created           5
approved          2
Name: count, dtype: int64


In [4]:
# Calculate delivery time in days (how long from purchase to delivery)
orders_delivered['delivery_time_days'] = (
    orders_delivered['order_delivered_customer_date'] - 
    orders_delivered['order_purchase_timestamp']
).dt.days

# Calculate if order was delivered late (delivered after estimated date)
orders_delivered['is_late'] = (
    orders_delivered['order_delivered_customer_date'] > 
    orders_delivered['order_estimated_delivery_date']
).astype(int)

# Extract useful time parts from purchase date
orders_delivered['order_year']    = orders_delivered['order_purchase_timestamp'].dt.year
orders_delivered['order_month']   = orders_delivered['order_purchase_timestamp'].dt.month
orders_delivered['order_day']     = orders_delivered['order_purchase_timestamp'].dt.day
orders_delivered['order_dayofweek'] = orders_delivered['order_purchase_timestamp'].dt.day_name()

# Confirm new columns
print("New columns added ✓")
print()
print(f"Average delivery time:  {orders_delivered['delivery_time_days'].mean():.1f} days")
print(f"Longest delivery:       {orders_delivered['delivery_time_days'].max()} days")
print(f"Shortest delivery:      {orders_delivered['delivery_time_days'].min()} days")
print()
print(f"Late orders:            {orders_delivered['is_late'].sum():,}")
print(f"On time orders:         {(orders_delivered['is_late']==0).sum():,}")
print(f"Late order rate:        {orders_delivered['is_late'].mean()*100:.1f}%")

New columns added ✓

Average delivery time:  12.1 days
Longest delivery:       209.0 days
Shortest delivery:      0.0 days

Late orders:            7,826
On time orders:         88,652
Late order rate:        8.1%


In [5]:
# Fill missing category names with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Fill missing dimensions with the median value of each column
products['product_weight_g']   = products['product_weight_g'].fillna(products['product_weight_g'].median())
products['product_length_cm']  = products['product_length_cm'].fillna(products['product_length_cm'].median())
products['product_height_cm']  = products['product_height_cm'].fillna(products['product_height_cm'].median())
products['product_width_cm']   = products['product_width_cm'].fillna(products['product_width_cm'].median())

# Confirm no more missing values
print("Products table cleaned ✓")
print()
print("Missing values remaining:")
print(products.isnull().sum())

Products table cleaned ✓

Missing values remaining:
product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64


In [6]:
# Merge English category names into products table
products = products.merge(
    categories,
    on='product_category_name',
    how='left'
)

# Fill any categories that had no translation with 'unknown'
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

# Confirm
print("English category names added ✓")
print()
print(f"Total products:        {len(products):,}")
print()
print("Sample categories in English:")
print(products['product_category_name_english'].value_counts().head(10))

English category names added ✓

Total products:        32,951

Sample categories in English:
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [7]:
# Start with delivered orders
master = orders_delivered.copy()

# Add customer details
master = master.merge(customers, on='customer_id', how='left')

# Add order items (price and freight)
master = master.merge(order_items, on='order_id', how='left')

# Add product details
master = master.merge(products[['product_id',
                                'product_category_name_english',
                                'product_weight_g']], 
                      on='product_id', how='left')

# Add seller details
master = master.merge(sellers, on='seller_id', how='left')

# Add payment details
master = master.merge(payments[['order_id',
                                'payment_type',
                                'payment_installments',
                                'payment_value']],
                      on='order_id', how='left')

# Confirm
print("Master table built ✓")
print()
print(f"Rows:     {len(master):,}")
print(f"Columns:  {master.shape[1]}")
print()
print("Column names:")
print(list(master.columns))

Master table built ✓

Rows:     115,038
Columns:  32

Column names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_time_days', 'is_late', 'order_year', 'order_month', 'order_day', 'order_dayofweek', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name_english', 'product_weight_g', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_type', 'payment_installments', 'payment_value']


In [8]:
# Check for any remaining missing values in key columns
print("=== Missing values in master table ===")
print(master.isnull().sum()[master.isnull().sum() > 0])

print()

# Save master table to cleaned folder
master.to_csv("../data/cleaned/master_orders.csv", index=False)

print("Master table saved ✓")
print()

# Confirm the file was saved
import os
file_size = os.path.getsize("../data/cleaned/master_orders.csv")
print(f"File size: {file_size / 1024 / 1024:.1f} MB")
print()
print("=" * 50)
print("PHASE 2 COMPLETE ✓")
print(f"Clean master table: 115,038 rows x 32 columns")
print(f"Saved to: data/cleaned/master_orders.csv")
print("=" * 50)

=== Missing values in master table ===
order_approved_at                15
order_delivered_carrier_date      2
order_delivered_customer_date     8
delivery_time_days                8
payment_type                      3
payment_installments              3
payment_value                     3
dtype: int64

Master table saved ✓

File size: 44.6 MB

PHASE 2 COMPLETE ✓
Clean master table: 115,038 rows x 32 columns
Saved to: data/cleaned/master_orders.csv


In [9]:
# Add review_score to master table
reviews_clean = reviews[['order_id', 'review_score']].dropna()
reviews_clean = reviews_clean.drop_duplicates(subset='order_id')

# Merge into master
master = pd.read_csv("../data/cleaned/master_orders.csv")
master = master.merge(reviews_clean, on='order_id', how='left')

# Save updated master table
master.to_csv("../data/cleaned/master_orders.csv", index=False)

print(f"review_score added ✓")
print(f"Columns now: {master.shape[1]}")
print(f"Missing review scores: {master['review_score'].isnull().sum():,}")
print(f"Review score distribution:")
print(master['review_score'].value_counts().sort_index())

review_score added ✓
Columns now: 33
Missing review scores: 861
Review score distribution:
review_score
1.0    13003
2.0     3868
3.0     9588
4.0    22029
5.0    65689
Name: count, dtype: int64


In [10]:
import sqlite3

db_path = "C:/Users/USER-PC/OneDrive/Desktop/olist-ecommerce-project/olist_database.db"
conn = sqlite3.connect(db_path)

master.to_sql(
    name='master_orders',
    con=conn,
    if_exists='replace',
    index=False
)

check = pd.read_sql("SELECT COUNT(*) as rows FROM master_orders", conn)
print("Database updated with review_score ✓")
print(f"Rows in database: {check['rows'][0]:,}")
conn.close()
print("Done ✓")

Database updated with review_score ✓
Rows in database: 115,038
Done ✓
